# Advanced Prompting: Chain of Thought and ReAct (Reasoning + Acting)

This notebook is based off the **Chain of Thought and ReAct (Reasoning + Acting)** notebook found in the [applied-ai-engieering-examples](https://github.com/GoogleCloudPlatform/applied-ai-engineering-samples) GitHub repository. This repository contains reference guides, blueprints, code samples, and hands-on labs developed by the Google Cloud Applied AI Engineering team.

# Introduction

The target audience of this notebook are engineering prompts to repeatedly execute a task, workflow, process, function, etc. Stability and performance are more important than when prompting for a one-off need.

This notebook covers two powerful LLM prompting strategies: **Chain of Thought** and **ReAct** (Reasoning + Acting).

ReAct (and its variants) are the current state-of-the-art prompting technique to improve LLM reasoning while minimizing hallucinations.

The four parts of this notebook are are:

1.  Chain-of-Thought Prompting: Using language descriptions of reasoning to improve LLM outputs.
2.  Actions, Retrieval, and Tool Use: How LLMs interact with external systems.
3.  ReAct (Reasoning + Acting) Prompting: Combining the written reasoning descriptions of chain-of-thought prompting with external system interactions.

This notebook was tested in Colab.

## Prerequisites

-   An understanding of LLMs (large language models):
    -   What an LLM is and how they work.
    -   LLMs as repetitive next-token predictors.
    -   LLM predictions maximize resemblance to the training data.
-   Experience with LLM prompting:
    -   What it means to "prompt" a language model. [Recommended resource](https://cloud.google.com/vertex-ai/docs/generative-ai/learn/introduction-prompt-design).
    -   The difference between [zero-shot, one-shot, and few-shot](https://cloud.google.com/vertex-ai/docs/generative-ai/learn/introduction-prompt-design#include-examples) prompting, and an understanding why few-shot prompting is essential for maximizing performance and robustness.
-   Basic familiarity with Google Cloud Vertex LLMs. [Recommended resource](https://cloud.google.com/vertex-ai/docs/generative-ai/start/quickstarts/api-quickstart)

## Key Terminology

For consistency this notebook uses the following terms in specific ways:

*   **Prompt**: A templated LLM call, created using specific techniques that maximize the performance and robustness of the call regardless of what values are inserted into the template.
*   **LLM Call**: Sending text to an LLM.
*   **LLM Response**: Text predicted by the LLM, what comes back from the LLM when making an LLM call.
*   **Chain/Chaining** Depending on context:
    *   In chain-of-thought prompting, logically sequential steps of reasoning.
    *   In LLM systems, sequential calls to an LLM, where each call depends on a previous call's response.
*   **Exemplar**: An "example" in a one- or few-shot prompt.
    *   Used to avoid confusion with "example" in the traditional ML sense, i.e., "a piece of data" (as in "training examples").

## References

*   Kojima, Takeshi, et al. "Large language models are zero-shot reasoners." Advances in neural information processing systems 35 (2022): 22199-22213. [Link](https://arxiv.org/abs/2205.11916) (accessed 2023 09 22)
*   Wang, Xuezhi, et al. "Self-consistency improves chain of thought reasoning in language models." arXiv preprint arXiv:2203.11171 (2022). [Link](https://arxiv.org/abs/2203.11171) (accessed 2023 09 03).
*   Wei, Jason, et al. "Chain-of-thought prompting elicits reasoning in large language models." Advances in Neural Information Processing Systems 35 (2022): 24824-24837. [Link](https://arxiv.org/abs/2201.11903) (accessed 2023 09 03).
*   Yao, Shunyu, et al. "React: Synergizing reasoning and acting in language models." arXiv preprint arXiv:2210.03629 (2022). [Link](https://arxiv.org/abs/2210.03629) (accessed 2023 09 03).


## Setup -- Run This Code First!


In [1]:
# Tested with these package versions.
!python -m pip install --user google-cloud-aiplatform==1.71.1 prettyprinter==0.18.0 wikipedia==1.4.0 numexpr

  Preparing metadata (setup.py) ... done
INFO: pip is looking at multiple versions of grpcio-status to determine which version is compatible with other requirements. This could take a while.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.2/6.2 MB 37.0 MB/s  0:00:00
  DEPRECATION: Building 'wikipedia' using the legacy setup.py bdist_wheel mechanism, which will be removed in a future version. pip 25.3 will enforce this behaviour change. A possible replacement is to use the standardized build interface by setting the `--use-pep517` option, (possibly combined with `--no-build-isolation`), or adding a `pyproject.toml` file to the source tree of 'wikipedia'. Discussion can be found at https://github.com/pypa/pip/issues/6334
  Created wheel for wikipedia: filename=wikipedia-1.4.0-py3-none-any.whl size=11757 sha256=002a426c883826b56540335d415646388a478d5befabe3d29dec73506fd626d8
  Stored in directory: /home/jupyter/.cache/pip/wheels/5e/b6/c5/93f3dec388ae76edc830cb42901bb0232504dfc0df02fc50de
Su

In [2]:
## Automatically restart kernel after installs so that your environment can access the new packages
import IPython

app = IPython.Application.instance()
app.kernel.do_shutdown(True)

{'status': 'ok', 'restart': True}

**Set your Google Cloud project ID and Location in the next cell.**


In [1]:
PROJECT_ID = "qwiklabs-gcp-03-76792ab47eed"  # @param {type:"string"}
LOCATION = "us-central1"  # @param {type:"string"}
# Code examples may misbehave if the model is changed.
MODEL_NAME = "gemini-2.0-flash-001"

# Set up Vertex AI.
import vertexai
from vertexai.generative_models import GenerativeModel, GenerationConfig

vertexai.init(project=PROJECT_ID,
              location=LOCATION)
parameters = {
    "temperature": 0,
    "max_output_tokens": 1024,
    "top_p": 0.8,
    "top_k": 40
}

model = GenerativeModel(MODEL_NAME)

In [2]:
# This function is used throughout the notebook to show the full LLM call and the response.
import time
from google.api_core.exceptions import ResourceExhausted

def call_llm(model, parameters, llm_call, show_activity=True, max_retries=5):
    
    generation_config = GenerationConfig(**parameters)
    
    attempt = 1
    
    while attempt <= max_retries:
    
        try:
            output = model.generate_content(llm_call, generation_config=generation_config)
            response = str(output.candidates[0].content.parts[0]).split("text:")[1]

            if show_activity:
                BOLD = "\033[1m"
                UNFORMAT = "\033[0m\x1B[0m"
                print(f"{BOLD}The call to the LLM:{UNFORMAT}\n{llm_call}\n")
                print(f"{BOLD}The response:{UNFORMAT}\n{response}")

            return response  # Return to `_` if not needed.

        except ResourceExhausted as e:
            time.sleep(2**attempt)
            attempt += 1
            if attempt > max_retries:
                raise ResourseExhausted(e)
            continue
# Wrap code cell output to improve notebook readability.
# Source: https://stackoverflow.com/questions/58890109/line-wrapping-in-collaboratory-google-results/61401455#61401455
from IPython.core.formatters import BaseFormatter
from IPython.display import HTML, display

class MultilineStringFormatter(BaseFormatter):
    def __call__(self, obj):
        if isinstance(obj, str) and '\n' in obj:
            return f'<pre>{obj}</pre>'
        return None

# Register the custom formatter
ip = get_ipython()
ip.display_formatter.formatters['text/html'].for_type(str, MultilineStringFormatter())

def set_css(arg):
  display(HTML('''
  <style>
    pre {
        white-space: pre-wrap;
    }
  </style>
  '''))
get_ipython().events.register('pre_run_cell', set_css)

# Part 1: Chain-of-Thought Prompting

To LLMs, chains are more than a fashionable accessory. The technique introduced in this [paper](https://arxiv.org/pdf/2201.11903) is a novel approach to enhance the reasoning capabilities of Large Language Models (LLMs), especially in multi-step reasoning tasks.

## Overview



In chain-of-thought prompting, you provide one- or few-shot exemplars showing the reasoning steps to get to a desired output. This is different from standard one- or few-shot prompting, where your exemplars show only the input and the correct output.

The reasoning breakdown you provide in chain-of-thought exemplars is similar to the natural language internal monologue a person has as they think through a problem or task.

If "internal monologue" is a strange concept, think about how you verbalize your thoughts to solve a problem or accomplish a task. For example, you're cooking dinner:

```Ok I've chopped the celery. Now I need to get started on the chicken. Is the oven on? Let me start preheating the oven. Wait, what temperature? I need to check the recipe again...```

This "internal monologue" or "inner speech" facilitates applying problem solving patterns to new problems we haven't seen before, by identifying what should happen next to make progress on the task.

By calling the LLM with exemplars that include an "internal monologue" of text reasoning, the LLM produces responses that include similar text reasoning. Having the LLM generate the reasoning text as part of the response increases the chance the response ends with the desired output.

The reasoning steps in the response also provide interpretability of how the LLM arrived at the final output.


## Chain of Thought Basics

Questions about space exploration are a good chain-of-thought demonstration, since they can require multiple steps of reasoning.


In [3]:
question = """
  Q: The Perseverance rover was launched on July 30, 2020, and landed on Mars on February 18, 2021. The journey took approximately 7 months. If its primary mission is scheduled to last one Martian year, which is about 2 Earth years, in what year is the primary mission expected to conclude?
  A: The answer is 2023.
  Q: A new satellite is scheduled to launch in 2026. Its construction phase takes 3 years, and its operational phase is planned for 5 years. In what year will the satellite be decommissioned?
  A:
"""
_ = call_llm(model, parameters, question)

The call to the LLM:

  Q: The Perseverance rover was launched on July 30, 2020, and landed on Mars on February 18, 2021. The journey took approximately 7 months. If its primary mission is scheduled to last one Martian year, which is about 2 Earth years, in what year is the primary mission expected to conclude?
  A: The answer is 2023.
  Q: A new satellite is scheduled to launch in 2026. Its construction phase takes 3 years, and its operational phase is planned for 5 years. In what year will the satellite be decommissioned?
  A:


The response:
 "The satellite will be decommissioned in 2031.\n"



Rewriting the prompt to include a chain of thought shows the LLM how to decompose the question into multiple simple steps of reasoning.

The model response then follows a similar chain of thought, increasing the likelihood and better explanation of the answer.

In [5]:
question = """
  Q: The Perseverance rover was launched on July 30, 2020, and landed on Mars on February 18, 2021. The journey took approximately 7 months. If its primary mission is scheduled to last one Martian year, which is about 2 Earth years, in what year is the primary mission expected to conclude?
  A: The rover landed in 2021. The primary mission is 2 Earth years long. 2021 + 2 = 2023. The answer is 2023.
  Q: A new satellite is scheduled to launch in 2026. Its construction phase takes 3 years, and its operational phase is planned for 5 years. In what year will the satellite be decommissioned?
  A:
"""
_ = call_llm(model, parameters, question)

The call to the LLM:

  Q: The Perseverance rover was launched on July 30, 2020, and landed on Mars on February 18, 2021. The journey took approximately 7 months. If its primary mission is scheduled to last one Martian year, which is about 2 Earth years, in what year is the primary mission expected to conclude?
  A: The rover landed in 2021. The primary mission is 2 Earth years long. 2021 + 2 = 2023. The answer is 2023.
  Q: A new satellite is scheduled to launch in 2026. Its construction phase takes 3 years, and its operational phase is planned for 5 years. In what year will the satellite be decommissioned?
  A:


The response:
 "The satellite launches in 2026 and has a 5-year operational phase. 2026 + 5 = 2031. The answer is 2031.\n"



The LLM response will usually mimic the reasoning style in the exemplars. This means you'll get the best performance if the chain of thought reasoning in your exemplars is a good fit for the task.

Compare the cells below. The first example uses a simple exemplar that is a poor match for the more complex question, leading to an poor match.

In [6]:
# The correct answer is 105 GB today, 82.5 GB tomorrow.
question = """
A high-gain orbiter can transmit 50 GB of data per day.
A medium-gain orbiter can transmit 20 GB of data per day.
A low-gain orbiter can transmit 5 GB of data per day.
A Mars mission has 3 orbiters. 2 are high-gain, 1 is low-gain.
Tomorrow, mission control will switch one high-gain orbiter to medium-gain to conserve power.
And the low-gain orbiter is hit by a micrometeoroid, cutting its transmission rate in half.
How much data can be transmitted today? How much tomorrow?
"""

_ = call_llm(model, parameters, question)

The call to the LLM:

A high-gain orbiter can transmit 50 GB of data per day.
A medium-gain orbiter can transmit 20 GB of data per day.
A low-gain orbiter can transmit 5 GB of data per day.
A Mars mission has 3 orbiters. 2 are high-gain, 1 is low-gain.
Tomorrow, mission control will switch one high-gain orbiter to medium-gain to conserve power.
And the low-gain orbiter is hit by a micrometeoroid, cutting its transmission rate in half.
How much data can be transmitted today? How much tomorrow?


The response:
 "Here\'s the breakdown:\n\n**Today\'s Data Transmission:**\n\n*   High-gain (2 orbiters): 2 * 50 GB/day = 100 GB/day\n*   Low-gain (1 orbiter): 1 * 5 GB/day = 5 GB/day\n*   Total today: 100 GB/day + 5 GB/day = **105 GB/day**\n\n**Tomorrow\'s Data Transmission:**\n\n*   High-gain (1 orbiter): 1 * 50 GB/day = 50 GB/day\n*   Medium-gain (1 orbiter): 1 * 20 GB/day = 20 GB/day\n*   Low-gain (1 orbiter, reduced): 1 * (5 GB/day / 2) = 2.5 GB/day\n*   Total tomorrow: 50 GB/day + 20 GB/day

In [7]:
# This exemplar is about a one-time calculation and doesn't fit the new question well.
one_shot_exemplar = """Q: The James Webb Space Telescope has 18 primary mirror segments.
Each segment is coated with 48 grams of gold.
How much gold is on the primary mirror in total?
A: There are 18 segments. Each has 48 grams of gold.
18 * 48 = 864. The answer is 864 grams.
Q: """

llm_call = f"{one_shot_exemplar}{question}\nA:"
_ = call_llm(model, parameters, llm_call)


The call to the LLM:
Q: The James Webb Space Telescope has 18 primary mirror segments.
Each segment is coated with 48 grams of gold.
How much gold is on the primary mirror in total?
A: There are 18 segments. Each has 48 grams of gold.
18 * 48 = 864. The answer is 864 grams.
Q: 
A high-gain orbiter can transmit 50 GB of data per day.
A medium-gain orbiter can transmit 20 GB of data per day.
A low-gain orbiter can transmit 5 GB of data per day.
A Mars mission has 3 orbiters. 2 are high-gain, 1 is low-gain.
Tomorrow, mission control will switch one high-gain orbiter to medium-gain to conserve power.
And the low-gain orbiter is hit by a micrometeoroid, cutting its transmission rate in half.
How much data can be transmitted today? How much tomorrow?

A:

The response:
 "Here\'s the breakdown of the data transmission calculations:\n\n**Today:**\n\n*   **High-Gain Orbiters:** 2 orbiters * 50 GB/orbiter = 100 GB\n*   **Low-Gain Orbiter:** 1 orbiter * 5 GB/orbiter = 5 GB\n*   **Total Today:** 1

The output may have mistakes or even the process itself also is not the best way. The LLM response fails to account for all the details. For this task, it's better to use a chain of thought exemplar that is more similar in structure to the question.

In [8]:
# This exemplar shows a more complex, multi-step calculation that is a better fit.
better_one_shot_exemplar = """Q: Waymo has 2 self-driving car models.
The 'Firefly' model has 2 LiDAR sensors.
The 'Jaguar' model has 5 LiDAR sensors.
Today, the fleet has 10 Firefly cars and 20 Jaguar cars.
Tomorrow, they are retiring 3 Firefly cars and adding 5 more Jaguar cars.
How many total LiDAR sensors are in the fleet today? How many tomorrow?
A: Today's Firefly sensors: 10 cars * 2 sensors/car = 20 sensors.
Today's Jaguar sensors: 20 cars * 5 sensors/car = 100 sensors.
Today the fleet has 20 + 100 = 120 sensors.
Tomorrow, there will be 10 - 3 = 7 Firefly cars.
Tomorrow, there will be 20 + 5 = 25 Jaguar cars.
Tomorrow's Firefly sensors: 7 cars * 2 sensors/car = 14 sensors.
Tomorrow's Jaguar sensors: 25 cars * 5 sensors/car = 125 sensors.
Tomorrow the fleet will have 14 + 125 = 139 sensors.
Q: """

# This is the same question as before.
question = """
A high-gain orbiter can transmit 50 GB of data per day.
A medium-gain orbiter can transmit 20 GB of data per day.
A low-gain orbiter can transmit 5 GB of data per day.
A Mars mission has 3 orbiters. 2 are high-gain, 1 is low-gain.
Tomorrow, mission control will switch one high-gain orbiter to medium-gain to conserve power.
And the low-gain orbiter is hit by a micrometeoroid, cutting its transmission rate in half.
How much data can be transmitted today? How much tomorrow?
"""

llm_call = f"{better_one_shot_exemplar}{question}\nA:"
_ = call_llm(model, parameters, llm_call)


The call to the LLM:
Q: Waymo has 2 self-driving car models.
The 'Firefly' model has 2 LiDAR sensors.
The 'Jaguar' model has 5 LiDAR sensors.
Today, the fleet has 10 Firefly cars and 20 Jaguar cars.
Tomorrow, they are retiring 3 Firefly cars and adding 5 more Jaguar cars.
How many total LiDAR sensors are in the fleet today? How many tomorrow?
A: Today's Firefly sensors: 10 cars * 2 sensors/car = 20 sensors.
Today's Jaguar sensors: 20 cars * 5 sensors/car = 100 sensors.
Today the fleet has 20 + 100 = 120 sensors.
Tomorrow, there will be 10 - 3 = 7 Firefly cars.
Tomorrow, there will be 20 + 5 = 25 Jaguar cars.
Tomorrow's Firefly sensors: 7 cars * 2 sensors/car = 14 sensors.
Tomorrow's Jaguar sensors: 25 cars * 5 sensors/car = 125 sensors.
Tomorrow the fleet will have 14 + 125 = 139 sensors.
Q: 
A high-gain orbiter can transmit 50 GB of data per day.
A medium-gain orbiter can transmit 20 GB of data per day.
A low-gain orbiter can transmit 5 GB of data per day.
A Mars mission has 3 orbiter

# Part 2: Actions, Retrieval, and Tool Use

## Hallucinations, Grounding, and Tools/Actions/Retrieval/RAG

LLMs are not reliable sources of facts. When factual accuracy is important, relying on an LLM's internal knowledge is risky. When an LLM response is factually incorrect it is often called a "hallucination".

The best way to manage hallucinations is to connect an LLM to an accurate and up-to-date external data source. This process is often called "grounding" or "Retrieval Augmented Generation" (RAG).

See what output this LLM call gives for a specific technical question:

In [10]:
question = "What rocket was used to launch the Galileo spacecraft to Jupiter?"
_ = call_llm(model, parameters, question)

The call to the LLM:
What rocket was used to launch the Galileo spacecraft to Jupiter?

The response:
 "The Galileo spacecraft was launched to Jupiter using the **Space Shuttle Atlantis (STS-34)** as its first stage, and then an **Inertial Upper Stage (IUS)** to boost it out of Earth orbit and towards Jupiter.\n"



The model may answer correctly but sometimes it can give a poor result. To ensure accuracy, we can give the model a "tool" to look up the answer.

## How LLM Tool Use Works: From Simple Lookup to Complex Explanation

Imagine you want to understand a complex, cutting-edge scientific concept. Asking an LLM directly might give you a definition that's five years old, which in a field like quantum computing, is ancient history.

Instead, we guide the LLM through a two-step process to answer the question: "Explain the concept of 'quantum supremacy' and mention a company that has claimed to achieve it."

1. **First Conversation:** "What is the best search term for this?"

* **You to the LLM:** "I need to understand 'quantum supremacy' and who's achieved it. Your job is to give me the most precise search term for a Wikipedia lookup. Just the search term, nothing else."
* **LLM to you:** "quantum supremacy"

2. **Second Conversation:** "Here is a current article. Now, explain it to me."

* You take that search term, use a tool to fetch the latest Wikipedia article, and bring it back to the LLM.
* **You to the LLM:** "Okay, I have retrieved the latest information. Here is the up-to-date Wikipedia article: [You paste the full text from the Wikipedia article here]. Now, using only this article, please explain the concept of 'quantum supremacy' and mention a company that has claimed to achieve it."
* **LLM to you:** "Quantum supremacy is the goal of demonstrating that a programmable quantum computer can solve a problem that no classical computer can solve in any feasible amount of time. In 2019, Google announced it had achieved this with its Sycamore processor."

This two-step chain is incredibly effective. It elevates the LLM from a "know-it-all" to a "research assistant." It uses the LLM's language skills to formulate the perfect query (Step 1) and then uses its summarization and reasoning skills to answer the question based only on the fresh, factual data you provide (Step 2).

In [11]:
import wikipedia
def wiki_tool(query, return_chars = 1000):
    try:
        page = wikipedia.page(query, auto_suggest=False, redirect=True).content
    # If no exact match, take Wikipedia's auto-suggestion.
    except wikipedia.exceptions.PageError as e:
        page = wikipedia.page(query, auto_suggest=True, redirect=True).content
    snippet = page[0:return_chars]
    return snippet

## Chaining LLM Calls for Tool Use: A Step-by-Step Guide
Let's walk through our quantum supremacy question. Answering this correctly requires a precise definition and recent, event-based knowledge that an LLM might not have.

### Step 1: Building the Prompt for the First LLM Call
To get the LLM to generate a clean search query, we give it clear instructions and a high-quality example. This is called **one-shot prompting**.

In [12]:
# The context tells the LLM its role and the format to use.
context = """Your task is to answer questions using a lookup of Wikipedia. 
You will be given a question. Your job is to generate the best possible search query for Wikipedia to find the answer.
Write the search query and then write '<STOP>'."""

# The exemplar shows the LLM a perfect example.
# This teaches it to extract the core concept from a question.
exemplar = """Question: Can you explain how blockchain technology works?
Wikipedia Search: blockchain<STOP>"""

_ = call_llm(model, parameters, question)

The call to the LLM:
What rocket was used to launch the Galileo spacecraft to Jupiter?

The response:
 "The Galileo spacecraft was launched to Jupiter using the **Space Shuttle Atlantis (STS-34)** as its first stage, and then an **Inertial Upper Stage (IUS)** to boost it out of Earth orbit and towards Jupiter.\n"



### Step 2: Making the First Call (Generating the Search Query)

Now, we combine the context, the exemplar, and our new, complex question into a single prompt for the LLM.

In [13]:
# The <STOP> token is a crucial instruction for our program, telling it precisely where the search query ends.
question = "Explain the concept of 'quantum supremacy' and mention a company that has claimed to achieve it."

# Assemble the full prompt for the first call
step_one_prompt = f"""{context}

{exemplar}

Question: {question}
Wikipedia Search:"""

# We send this to the LLM. The ideal response is clean and simple.
# For demonstration, let's assume the LLM correctly responds:
step_one_response = "quantum supremacy<STOP>"

print(f"LLM's Ideal Response: {step_one_response}")

LLM's Ideal Response: quantum supremacy<STOP>


### Step 3: Using the Tool

Our program parses the response to get the query (quantum supremacy) and feeds it into our wiki_tool to retrieve the real-time information.

In [15]:
# The hypothetical text our wiki_tool would retrieve.
# I've included key phrases that would be in a real article.
wiki_text = """
In quantum computing, quantum supremacy or quantum advantage is the goal of demonstrating that a programmable quantum device can solve a problem that no classical computer can solve in any feasible amount of time. The term was coined by John Preskill in 2012.

Conceptually, this involves finding a problem that is computationally difficult for classical computers but is amenable to a quantum computer. A notable property of quantum supremacy is that the problem does not need to be useful, so it is primarily a scientific milestone.

In October 2019, Google AI, with its Sycamore processor, published a paper in Nature claiming to have achieved quantum supremacy. They reported that their processor performed a specific task in 200 seconds that would have taken the world's most advanced classical supercomputer an estimated 10,000 years to complete. This claim was later contested by other companies like IBM, who argued that a different classical algorithm could solve the problem in a matter of days.
"""

# In a real application, this text would come from our live tool call:
# wiki_query = get_wiki_query(step_one_response)
# wiki_text = wiki_tool(wiki_query)

print(f"Snippet Retrieved from Wikipedia:\n{wiki_text}")

Snippet Retrieved from Wikipedia:

In quantum computing, quantum supremacy or quantum advantage is the goal of demonstrating that a programmable quantum device can solve a problem that no classical computer can solve in any feasible amount of time. The term was coined by John Preskill in 2012.

Conceptually, this involves finding a problem that is computationally difficult for classical computers but is amenable to a quantum computer. A notable property of quantum supremacy is that the problem does not need to be useful, so it is primarily a scientific milestone.

In October 2019, Google AI, with its Sycamore processor, published a paper in Nature claiming to have achieved quantum supremacy. They reported that their processor performed a specific task in 200 seconds that would have taken the world's most advanced classical supercomputer an estimated 10,000 years to complete. This claim was later contested by other companies like IBM, who argued that a different classical algorithm coul

### Step 4: Making the Second Call (Answering the Question with Grounded Context)

This is the final and most important step. We construct a new, comprehensive prompt that includes all the information gathered so far. This gives the LLM everything it needs to construct a high-quality, factual answer.

In [16]:
# We assemble the entire history of the interaction into a final prompt.
step_two_prompt = f"""{context}

{exemplar}

Question: {question}
Wikipedia Search: {step_one_response}

Wikipedia Article: {wiki_text}
Answer:"""

# The LLM's task is now to synthesize an answer from the provided text.
# The ideal response would be:
step_two_response = """Quantum supremacy, or quantum advantage, is the goal of showing that a quantum computer can solve a problem that a classical computer cannot solve in a realistic amount of time. [2] In October 2019, Google AI claimed to have achieved this milestone with their Sycamore processor, stating their quantum computer solved a problem in 200 seconds that would take a supercomputer 10,000 years. [4, 5, 8]"""

print(f"Full Prompt for Second Call:\n{step_two_prompt}")
print("---")
print(f"LLM's Final Grounded Answer:\n{step_two_response}")

Full Prompt for Second Call:
Your task is to answer questions using a lookup of Wikipedia. 
You will be given a question. Your job is to generate the best possible search query for Wikipedia to find the answer.
Write the search query and then write '<STOP>'.

Question: Can you explain how blockchain technology works?
Wikipedia Search: blockchain<STOP>

Question: Explain the concept of 'quantum supremacy' and mention a company that has claimed to achieve it.
Wikipedia Search: quantum supremacy<STOP>

Wikipedia Article: 
In quantum computing, quantum supremacy or quantum advantage is the goal of demonstrating that a programmable quantum device can solve a problem that no classical computer can solve in any feasible amount of time. The term was coined by John Preskill in 2012.

Conceptually, this involves finding a problem that is computationally difficult for classical computers but is amenable to a quantum computer. A notable property of quantum supremacy is that the problem does not ne

## Why This Method Is Superior

By using a two-step tool chain, we transformed the task. Instead of asking the LLM to recall a complex definition from its vast but potentially outdated memory, we tasked it with a much more constrained and reliable job:

1. **Identify** the key topic in a question.
2. **Synthesize** an answer based only on a trusted, up-to-the-minute document.

This process of "Retrieve-then-Read" is fundamental to building reliable and factual AI systems. It allows us to combine the language fluency of an LLM with the accuracy of a real-time knowledge base, giving us the best of both worlds.


# Part 3: ReAct (Reasoning + Acting) Prompting

ReAct (reasoning + actions) combines chain of thought and tool usage together to reason through complex tasks by interacting with external systems. A ReAct chain has three interleaved parts:
- **Thoughts**: The LLM's plan or reasoning.
- **Actions**: LLM-generated commands to an external tool.
- **Observations**: The response from the external tool.

These steps are repeated until the LLM completes its task.

## Manually Running a ReAct Chain

This section runs a ReAct chain step-by-step. The exemplar is reframed to compare the launch dates of two famous space telescopes.

In [ ]:
context = """Answer questions with thoughts, actions, and observations.

Think about the next action to take. Then take an action.
All actions are a lookup of Wikipedia.
The Wikipedia action returns the beginning of the best-matching article.
When making a Wikipedia lookup action, end the lookup with <STOP>.
After the Wikipedia action, you will have an observation.
The observation is based on what you learn from the Wikipedia lookup action.
After the observation, begin the loop again with a thought.

Repeat as necessary a thought, taking an action, and having an observation.
Keep repeating as necessary until you know the answer to the question.
When you think you have an answer, return the answer in the format:
"Answer[answer goes here between square brackets]" as part of a thought.
Make sure to capitalize "Answer".

Only use information in the observations to answer the question."""

exemplar = """Example:
Question: Which was launched first, the James Webb Space Telescope or the Hubble Space Telescope?
Thought 1: I need to find the launch date for the James Webb Space Telescope.
Action 1: James Webb Space Telescope launch date<STOP>
Observation 1: The James Webb Space Telescope (JWST) is a space telescope designed primarily to conduct infrared astronomy. The most powerful telescope ever launched into space, its greatly improved infrared resolution and sensitivity allow it to view objects too old, distant, or faint for the Hubble Space Telescope. The telescope was launched on 25 December 2021 on an Ariane 5 rocket.
Thought 2: The James Webb Space Telescope was launched in 2021. Now I need to find the launch date for the Hubble Space Telescope.
Action 2: Hubble Space Telescope launch date<STOP>
Observation 2: The Hubble Space Telescope is a space telescope that was launched into low Earth orbit in 1990 and remains in operation. It was not the first space telescope, but it is one of the largest and most versatile, renowned both as a vital research tool and as a public relations boon for astronomy.
Thought 3: The Hubble Space Telescope was launched in 1990 and the James Webb Space Telescope was launched in 2021. 1990 is before 2021. Answer[Hubble Space Telescope]"""

_ = call_llm(model, parameters, question)


Let's try a new question that requires multiple lookups.

In [ ]:
question = "What was the initial purpose of the company that Google acquired in 2006, which is now a major video platform?"
llm_call_1 = f"{context}\n\n{exemplar}\n\nQuestion: {question}\nThought 1:"

response_1 = call_llm(model, parameters, llm_call_1)

Now we automate this process with a function.

## A Complete Python Code Snippet for Running ReAct Chains

The code snippet below automates the manual steps. It makes formatted ReAct calls to the LLM, extracts actions, executes them, and loops until it finds an answer.

In [ ]:
def wiki_react_chain(model,
                     parameters,
                     context,
                     exemplar,
                     question,
                     max_steps=7,
                     show_activity=False):
    # Call an LLM in a ReAct-style Thought -> Action -> Observation loop.
    next_llm_call = f"{context}\n\n{exemplar}\n\nQuestion: {question}\nThought 1:"

    step = 1
    while step <= max_steps:

        if show_activity:
            print(f"\033[1mReAct chain step {step}:\033[0m\x1B[0m")
        llm_response = call_llm(model, parameters, next_llm_call, show_activity)

        response_first_line = llm_response.splitlines()[0]
        first_line_answer_split = response_first_line.split("Answer[")
        if len(first_line_answer_split) > 1: # If there's a split on "Answer[".
            return first_line_answer_split[1].split("]")[0]

        # If no answer, assume following response line is action.
        response_second_line = llm_response.splitlines()[1]

        # Extract the wiki query from the action line of the response.
        wiki_query = response_second_line.split(":")[1].split("<STOP>")[0]
        wiki_query = wiki_query.strip()
        if show_activity:
            print(f"\033[1mQuerying wikipedia for: {wiki_query}.\033[0m\x1B[0m")
        wiki_text = wiki_tool(wiki_query)

        # Assemble the next LLM call.
        usable_response = f"{response_first_line}\n{response_second_line}"
        obs = f"Observation {step}: {wiki_text}"
        step += 1
        next_llm_call = f"{next_llm_call} {usable_response}\n{obs}\nThought {step}:"

    return None


## More ReAct Use Cases

The ReAct pattern can also be used for other tasks, like fact-checking. Here, we change the context and exemplar to guide the LLM to verify a claim about the planets.

In [ ]:
claim = "The diameter of Mars is smaller than the diameter of Mercury."

context_fact_check = """You are verifying claims as true or false.
Verify the claim with thoughts, actions, and observations.
Determine if there is an observation that SUPPORTS or REFUTES the claim.
When you think you have an answer, return the answer as "Answer[REFUTES]" or "Answer[SUPPORTS]".
Only use information in the observations to answer the question."""

exemplar_fact_check = """Example:
Claim: The James Webb Space Telescope was launched before the Hubble Space Telescope.
Thought 1: I need to find the launch date for the James Webb Space Telescope.
Action 1: James Webb Space Telescope launch date<STOP>
Observation 1: The Webb was launched on 25 December 2021...
Thought 2: The James Webb Space Telescope was launched in 2021. Now I need to find the launch date for the Hubble Space Telescope.
Action 2: Hubble Space Telescope launch date<STOP>
Observation 2: The Hubble Space Telescope (HST), launched by NASA on April 24, 1990...
Thought 3: The Hubble was launched in 1990 and the Webb was launched in 2021. The claim that Webb was launched before Hubble is false. Answer[REFUTES]"""

answer = wiki_react_chain(model,
                          parameters,
                          context_fact_check,
                          exemplar_fact_check,
                          claim,
                          show_activity = True)
print(f"\nFinal Answer: {answer}")